# Thesis Figures Notebook Skeleton

This notebook is a code-free structural skeleton derived from `thesis_figure_notebook_plan.md`.

## Purpose
- one section per thesis figure
- very short rationale for chosen format
- metadata block
- generation body structure only
- no implementation code yet

## Global notebook contract
- Planned notebook name: `thesis_figures.ipynb`
- Suggested export directory: `figures_generated/`
- Figure naming convention: `fig_1_1`, `fig_3_2`, `fig_5_3`, ...
- Primary thesis source: `co_om_thesis_enhanced.md`
- Empirical figures should prefer repo artifacts from `final_runs/**`
- Conceptual and hybrid figures should preserve thesis-faithful semantic checklists


## Shared setup structure

**Planned future cells**
1. imports
2. plotting theme / style
3. export helper
4. common path configuration
5. shared artifact loaders


In [ ]:
import os
import json
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".mplconfig"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch


In [ ]:
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.18,
    "grid.linestyle": "--",
})

FAMILY_COLORS = {
    "base-gnn": "#4c78a8",
    "multi-gnn": "#72b7b2",
    "memory-gnn": "#f58518",
}


In [ ]:
REPO_ROOT = Path.cwd()
THESIS_PATH = REPO_ROOT / "co_om_thesis_enhanced.md"
FIGURES_DIR = REPO_ROOT / "figures_generated"
FIGURES_DIR.mkdir(exist_ok=True)


In [ ]:
SPLIT_PATHS = {
    "5min": REPO_ROOT / "final_runs/5min-base-gnn/splits/split_summary.json",
    "1min": REPO_ROOT / "final_runs/1min-base-gnn-conv/splits/split_summary.json",
    "1sec": REPO_ROOT / "final_runs/1sec-base-gnn-conv/splits/split_summary.json",
}

REGIME_SPECS = {
    "5min": {"working_slice": (0.00, 0.90), "holdout_frac": 0.10, "lookback": "30 min = 6 bars", "horizon": "5 min = 1 bar"},
    "1min": {"working_slice": (0.00, 0.90), "holdout_frac": 0.10, "lookback": "30 min = 30 bars", "horizon": "5 min = 5 bars"},
    "1sec": {"working_slice": (0.50, 0.90), "holdout_frac": 0.225, "lookback": "2 min = 120 bars", "horizon": "2 min = 120 bars"},
}


In [ ]:
PRIMARY_BENCHMARK_SOURCES = {
    ("5min", "base-gnn-conv"): (REPO_ROOT / "final_runs/5min-base-gnn/final_report.csv", "adaptive_conv"),
    ("5min", "base-gnn-mpnn"): (REPO_ROOT / "final_runs/5min-base-gnn/final_report.csv", "adaptive_mpnn"),
    ("5min", "multi-gnn-conv"): (REPO_ROOT / "final_runs/5min-multi-gnn/final_report.csv", "dynamic_rel_conv"),
    ("5min", "multi-gnn-mpnn"): (REPO_ROOT / "final_runs/5min-multi-gnn/final_report.csv", "dynamic_edge_mpnn"),
    ("5min", "memory-gnn-conv"): (REPO_ROOT / "final_runs/5min-memory-gnn/final_report.csv", "conv"),
    ("5min", "memory-gnn-mpnn"): (REPO_ROOT / "final_runs/5min-memory-gnn/final_report.csv", "mpnn"),
    ("1min", "base-gnn-conv"): (REPO_ROOT / "final_runs/1min-base-gnn-conv/final_report.csv", None),
    ("1min", "base-gnn-mpnn"): (REPO_ROOT / "final_runs/1min-base-gnn-mpnn/final_report.csv", None),
    ("1min", "multi-gnn-conv"): (REPO_ROOT / "final_runs/1min-multi-gnn-conv/final_report.csv", None),
    ("1min", "multi-gnn-mpnn"): (REPO_ROOT / "final_runs/1min-multi-gnn-mpnn/final_report.csv", None),
    ("1min", "memory-gnn-conv"): (REPO_ROOT / "final_runs/1min-memory-gnn/final_report.csv", "conv"),
    ("1min", "memory-gnn-mpnn"): (REPO_ROOT / "final_runs/1min-memory-gnn/final_report.csv", "mpnn"),
    ("1sec", "base-gnn-conv"): (REPO_ROOT / "final_runs/1sec-base-gnn-conv/final_report.csv", None),
    ("1sec", "base-gnn-mpnn"): (REPO_ROOT / "final_runs/1sec-base-gnn-mpnn/final_report.csv", None),
    ("1sec", "multi-gnn-conv"): (REPO_ROOT / "final_runs/1sec-multi-gnn-conv/final_report.csv", None),
    ("1sec", "multi-gnn-mpnn"): (REPO_ROOT / "final_runs/1sec-multi-gnn-mpnn/final_report.csv", None),
    ("1sec", "memory-gnn-conv"): (REPO_ROOT / "final_runs/1sec-memory-gnn-conv/final_report.csv", None),
    ("1sec", "memory-gnn-mpnn"): (REPO_ROOT / "final_runs/1sec-memory-gnn-mpnn/final_report.csv", None),
}

def load_split_summary(freq):
    with open(SPLIT_PATHS[freq], "r", encoding="utf-8") as f:
        return json.load(f)


def load_primary_benchmark_table(model_state="last_cv"):
    rows = []
    for (freq, label), (csv_path, operator_filter) in PRIMARY_BENCHMARK_SOURCES.items():
        df = pd.read_csv(csv_path)
        df = df[df["model_state"] == model_state].copy()
        if operator_filter is not None and "operator" in df.columns:
            df = df[df["operator"] == operator_filter].copy()
        if df.empty:
            raise ValueError(f"No rows found for {freq} {label} ({model_state})")
        row = df.iloc[0].to_dict()
        row["frequency"] = freq
        row["model_label"] = label
        rows.append(row)
    return pd.DataFrame(rows)


## Figure 1.1 — Conceptual pipeline from limit order book snapshots to graph-based entry decisions

**Why this format**
- Hybrid is best because the figure is conceptual, but its objects are tightly tied to repo terminology and thesis pipeline stages.

**Metadata**
- Figure type: `hybrid`
- Evidence source: `mixed`
- Rendering method: `Prompt-first hybrid`
- Primary inputs: `co_om_thesis_enhanced.md`, `train_config.yaml`, `models/base_gnn_pipeline.py`, `models/multigraph_pipeline.py`, `models/memorygraph_pipeline.py`
- Acceptance check: ADA/BTC/ETH inputs, node features, relation-aware edges, graph prediction stage, entry decision, cost-aware final-holdout backtest


**Generation body — final image prompt**

```text
Create a clean academic systems diagram for a master's thesis in quantitative finance.

Title: "Conceptual pipeline from limit order book snapshots to graph-based entry decisions"

Style requirements:
- white background
- publication-quality vector-like look
- minimal color palette: dark blue, muted teal, gray, subtle orange accents
- professional academic layout, no marketing style
- landscape orientation
- sharp readable labels
- consistent arrow styles
- elegant, uncluttered composition

Diagram structure from left to right:
1. Three separate crypto limit order book snapshot panels labeled ADA, BTC, and ETH.
2. Feature extraction block with node and relation features.
3. Graph construction block with ADA, BTC, ETH and directed complete graph with self-loops.
4. Graph model block.
5. Output head block.
6. Decision block.
7. Evaluation block with realized exit, gross PnL, cost adjustment, final-holdout net PnL.

Important semantic constraint:
This is a controlled research benchmark pipeline from LOB snapshots to graph-based entry decisions and cost-aware backtest evaluation.
```


## Figure 1.2 — Research-question map for the controlled graph benchmark

**Why this format**
- Python is enough because the figure is essentially a structured mapping from RQ1–RQ4 to benchmark dimensions.

**Metadata**
- Figure type: `conceptual-executable`
- Evidence source: `thesis-only`
- Rendering method: `Python`
- Primary inputs: `co_om_thesis_enhanced.md`
- Acceptance check: RQ1→family, RQ2→Conv/MPNN, RQ3→temporal resolution, RQ4→`last_CV`/`final_refit`


**Generation body**
- Build a 4-row mapping diagram or matrix.
- Left column: `RQ1`–`RQ4`.
- Right side: benchmark dimensions with arrows / assignment cells.
- Keep the layout compact and thesis-like.

**Planned future code cells**
1. load thesis text / labels
2. prepare RQ-to-dimension structure
3. render figure
4. export figure


## Figure 3.1 — Graph input representation for the three-asset limit order book benchmark

**Why this format**
- Python is appropriate because the graph topology is explicit and can be rendered reproducibly, but annotations must emphasize relation semantics.

**Metadata**
- Figure type: `executable`
- Evidence source: `code-grounded`
- Rendering method: `Python`
- Primary inputs: `train_config.yaml`, `models/base_gnn_pipeline.py`, `models/multigraph_pipeline.py`, `models/memorygraph_pipeline.py`
- Acceptance check: ADA/BTC/ETH nodes, directed complete graph with self-loops, relation channels, node vs edge feature distinction


**Generation body**
- Use a fixed triangular node layout.
- Show self-loops explicitly.
- Label relation channels: `price_dep`, `order_flow`, `liquidity`.
- Add side annotation for node tensor vs relation-aware edge tensor.

**Planned future code cells**
1. load graph semantics from config/code notes
2. define topology and labels
3. render graph figure
4. export figure


## Figure 3.2 — Frequency regimes and final-holdout split design

**Why this format**
- Python is the right choice because the split structure exists in `split_summary.json` and should be reproduced from artifacts, not hand-drawn.

**Metadata**
- Figure type: `executable`
- Evidence source: `artifact-grounded`
- Rendering method: `Python`
- Primary inputs: `final_runs/5min-base-gnn/splits/split_summary.json`, `final_runs/1min-base-gnn-conv/splits/split_summary.json`, `final_runs/1sec-base-gnn-conv/splits/split_summary.json`
- Acceptance check: `5min`/`1min`/`1sec`, pre-holdout vs holdout, adapted 1sec regime, aligned final holdout intervals


**Generation body**
- One horizontal timeline per frequency.
- Show working slice, pre-holdout, final holdout.
- Annotate lookback / horizon / folds.
- Add note that `1sec` is a frequency-adapted stress test.

**Planned future code cells**
1. load split summaries
2. normalize timelines
3. render regime timelines
4. export figure


In [ ]:
split_summaries = {freq: load_split_summary(freq) for freq in SPLIT_PATHS}
regime_rows = []
for freq, summary in split_summaries.items():
    spec = REGIME_SPECS[freq]
    regime_rows.append({
        "frequency": freq,
        "work_start": spec["working_slice"][0],
        "work_end": spec["working_slice"][1],
        "holdout_start": spec["working_slice"][1] - spec["holdout_frac"],
        "holdout_end": spec["working_slice"][1],
        "lookback": spec["lookback"],
        "horizon": spec["horizon"],
        "cv_folds": summary["num_train_folds"],
        "purge_gap_bars": summary["purge_gap_bars"],
    })
regime_df = pd.DataFrame(regime_rows).sort_values("frequency", ascending=False)
regime_df


In [ ]:
regime_colors = {
    "outside": "#e5e7eb",
    "preholdout": "#4c78a8",
    "holdout": "#f58518",
}


In [ ]:
fig, axes = plt.subplots(
    2,
    1,
    figsize=(11.5, 6.6),
    constrained_layout=True,
    height_ratios=[1.25, 1],
)

ax = axes[0]
y_positions = np.arange(len(regime_df))[::-1]
bar_height = 0.58

for y, row in zip(y_positions, regime_df.to_dict("records")):
    ax.add_patch(Rectangle((0, y - bar_height / 2), 1.0, bar_height, facecolor=regime_colors["outside"], edgecolor="none", zorder=0))
    ax.add_patch(Rectangle((row["work_start"], y - bar_height / 2), row["work_end"] - row["work_start"], bar_height, facecolor=regime_colors["preholdout"], edgecolor="white", linewidth=0.8))
    ax.add_patch(Rectangle((row["holdout_start"], y - bar_height / 2), row["holdout_end"] - row["holdout_start"], bar_height, facecolor=regime_colors["holdout"], edgecolor="white", linewidth=0.8))
    ax.text(0.01, y, row["frequency"], va="center", ha="left", fontweight="bold")
    ax.text(0.915, y, f"holdout\n{row['holdout_end'] - row['holdout_start']:.1%}", va="center", ha="left", fontsize=8, color="#7c2d12")
    ax.text(
        row["work_start"] + 0.01,
        y - 0.43,
        f"lookback: {row['lookback']} | horizon: {row['horizon']} | folds: {row['cv_folds']} | purge gap: {row['purge_gap_bars']} bars",
        fontsize=8,
        ha="left",
        va="top",
        color="#334155",
    )

ax.set_xlim(0, 1.0)
ax.set_ylim(-0.75, len(regime_df) - 0.25)
ax.set_yticks([])
ax.set_xticks(np.linspace(0, 1, 6))
ax.set_xticklabels([f"{int(x * 100)}%" for x in np.linspace(0, 1, 6)])
ax.set_title("Figure 3.2 — Frequency regimes and final-holdout split design")
ax.set_xlabel("Full series position")
ax.text(0.0, len(regime_df) - 0.02, "Working sample and aligned holdout intervals", ha="left", va="bottom", fontsize=10, fontweight="bold")

ax2 = axes[1]
ax2.axis("off")
fold_x = [0.05, 0.44, 0.58, 0.72, 0.86]
widths = [0.34, 0.10, 0.10, 0.10, 0.09]
labels = ["train", "purge", "val", "purge", "test"]
fill = ["#4c78a8", "#cbd5e1", "#72b7b2", "#cbd5e1", "#54a24b"]
for x, w, lbl, color in zip(fold_x, widths, labels, fill):
    ax2.add_patch(Rectangle((x, 0.45), w, 0.18, facecolor=color, edgecolor="white"))
    ax2.text(x + w / 2, 0.54, lbl, ha="center", va="center", fontsize=9, fontweight="bold" if lbl in {"train", "val", "test"} else None)

ax2.annotate(
    "aligned final holdout used only after model development",
    xy=(0.83, 0.25),
    xytext=(0.48, 0.1),
    arrowprops=dict(arrowstyle="->", lw=1.2, color="#7c2d12"),
    color="#7c2d12",
    fontsize=9,
)
ax2.text(0.05, 0.78, "Representative purge-aware CV fold structure inside the pre-holdout region", fontsize=10, fontweight="bold")
ax2.text(0.05, 0.70, "The 5min and 1min regimes share the same task definition; the 1sec regime is an adapted high-frequency stress test.", fontsize=9)

fig


In [ ]:
fig_3_2_path = FIGURES_DIR / "fig_3_2_split_design.png"
fig.savefig(fig_3_2_path, bbox_inches="tight")
fig_3_2_path


## Figure 3.3 — Triple-barrier target construction for the ETH midpoint

**Why this format**
- Hybrid is best because the core path/barrier visualization can be built in Python, but the explanatory annotations need tight manual control.

**Metadata**
- Figure type: `hybrid`
- Evidence source: `mixed`
- Rendering method: `Python-primary`
- Primary inputs: `../Graph_Neural_Network_for_Market_Microstructure/dataset/ETH_1min.csv`, `../Graph_Neural_Network_for_Market_Microstructure/dataset/ETH_5min.csv`, `train_config.yaml`, `co_om_thesis_enhanced.md`
- Acceptance check: target timestamp, upper/lower/vertical barriers, realized exit, realized return, trade relevance, direction label, reproducible timestamp selection rule


**Generation body**
- Select one reproducible example timestamp with a clear barrier hit.
- Plot midpoint path after target time.
- Draw upper/lower/vertical barriers.
- Add labels for realized exit, realized return, trade label, direction label.

**Planned future code cells**
1. load ETH data and benchmark parameters
2. choose representative timestamp
3. render barrier figure
4. export figure
5. manual polish notes


## Figure 3.4 — Common entry-model backtest and post-cost PnL calculation

**Why this format**
- Hybrid fits best because the figure is a conceptual evaluation pipeline with formulas, not a raw empirical plot.

**Metadata**
- Figure type: `hybrid`
- Evidence source: `mixed`
- Rendering method: `Prompt-first hybrid`
- Primary inputs: `co_om_thesis_enhanced.md`, `train_config.yaml`
- Acceptance check: trade activation, direction choice, realized event exit, gross→net PnL with cost proxy


**Generation body — final image prompt**

```text
Create a clean academic flowchart for a quantitative finance thesis.

Title: "Common entry-model backtest and post-cost PnL calculation"

Flow structure:
1. model outputs with trade head and direction head
2. decision block
3. event-based holding block
4. gross PnL block
5. cost adjustment block
6. final net PnL output block

Semantic constraint:
This is a controlled benchmark backtest illustration, not a full execution simulator.
```


## Figure 3.5 — Purged walk-forward validation and deployment-oriented model states

**Why this format**
- Python is the strongest choice because both chronology and model-state comparison can be anchored to split artifacts and reported states.

**Metadata**
- Figure type: `executable`
- Evidence source: `artifact-grounded`
- Rendering method: `Python`
- Primary inputs: split summaries for `5min`, `1min`, `1sec`, plus thesis semantics
- Acceptance check: train/purge/validation/purge/test, chronological order, separate final holdout, visible `best_CV`, `last_CV`, `final_refit`


**Generation body**
- Draw fold chronology explicitly.
- Add a second annotation layer showing how each model state is obtained.
- Keep it distinct from Figure 3.2 by focusing on fold mechanics.

**Planned future code cells**
1. load split summaries and state semantics
2. build fold chronology representation
3. render validation/model-state diagram
4. export figure


In [ ]:
walkforward_summary = load_split_summary("1min")
preholdout = walkforward_summary["preholdout"]
holdout = walkforward_summary["holdout"]
cv_folds = walkforward_summary["cv_folds"]
last_fold = cv_folds[-1]

full_total = preholdout["n_samples"] + holdout["n_samples"]
preholdout_fraction = preholdout["n_samples"] / full_total
holdout_fraction = holdout["n_samples"] / full_total


In [ ]:
train_n = last_fold["train"]["n_samples"]
validation_n = last_fold["val"]["n_samples"]
test_n = last_fold["test"]["n_samples"]
purge_gap = walkforward_summary["purge_gap_bars"]
fold_total = train_n + validation_n + test_n + 2 * purge_gap

fold_segments = [
    ("train", train_n / fold_total, "#4c78a8"),
    ("purge", purge_gap / fold_total, "#cbd5e1"),
    ("validation", validation_n / fold_total, "#72b7b2"),
    ("purge", purge_gap / fold_total, "#cbd5e1"),
    ("test", test_n / fold_total, "#54a24b"),
]


In [ ]:
fig = plt.figure(figsize=(11.5, 7.2), constrained_layout=True)
grid = fig.add_gridspec(3, 1, height_ratios=[1, 1, 1.15])
ax1 = fig.add_subplot(grid[0])
ax2 = fig.add_subplot(grid[1])
ax3 = fig.add_subplot(grid[2])

# top panel: global experiment timeline
ax1.add_patch(Rectangle((0, 0.35), preholdout_fraction, 0.3, facecolor="#4c78a8", edgecolor="white"))
ax1.add_patch(Rectangle((preholdout_fraction, 0.35), holdout_fraction, 0.3, facecolor="#f58518", edgecolor="white"))
ax1.text(preholdout_fraction / 2, 0.5, "pre-holdout\n(model development)", ha="center", va="center", color="white", fontweight="bold")
ax1.text(preholdout_fraction + holdout_fraction / 2, 0.5, "final holdout\n(blind evaluation)", ha="center", va="center", color="white", fontweight="bold")
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)
ax1.set_yticks([])
ax1.set_xticks([0, preholdout_fraction, 1.0])
ax1.set_xticklabels(["start", "holdout begins", "end"])
ax1.set_title("Figure 3.5 — Purged walk-forward validation and deployment-oriented model states")
ax1.text(0.0, 0.86, "Experiment timeline (1min regime shown as representative chronological benchmark)", fontsize=10, fontweight="bold", ha="left")

# middle panel: representative final CV fold
ax2.axis("off")
current_x = 0.02
for label, width, color in fold_segments:
    scaled_width = width * 0.96
    ax2.add_patch(Rectangle((current_x, 0.36), scaled_width, 0.28, facecolor=color, edgecolor="white"))
    ax2.text(current_x + scaled_width / 2, 0.5, label, ha="center", va="center", fontsize=9, fontweight="bold" if label in {"train", "validation", "test"} else None)
    current_x += scaled_width
ax2.text(0.02, 0.82, f"Representative final CV fold | purge gap = {purge_gap} bars | folds = {walkforward_summary['num_train_folds']}", fontsize=10, fontweight="bold")
ax2.text(0.02, 0.16, "Chronology is preserved and leakage is reduced by inserting purge gaps around validation and test boundaries.", fontsize=9)

# bottom panel: model states
ax3.axis("off")
state_boxes = {
    "best_CV": (0.08, 0.58, 0.22, 0.22, "#72b7b2", "best_CV\nstrongest selected CV checkpoint"),
    "last_CV": (0.39, 0.58, 0.22, 0.22, "#4c78a8", "last_CV\nfinal chronological fold model"),
    "final_refit": (0.70, 0.58, 0.22, 0.22, "#f58518", "final_refit\nrefit on largest pre-holdout sample"),
}
for x, y, w, h, color, label in state_boxes.values():
    patch = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.03", facecolor=color, edgecolor="none", alpha=0.96)
    ax3.add_patch(patch)
    ax3.text(x + w / 2, y + h / 2, label, ha="center", va="center", fontsize=9, color="white", fontweight="bold")

ax3.annotate("", xy=(0.19, 0.58), xytext=(0.22, 0.38), arrowprops=dict(arrowstyle="->", lw=1.4, color="#475569"))
ax3.annotate("", xy=(0.50, 0.58), xytext=(0.50, 0.38), arrowprops=dict(arrowstyle="->", lw=1.4, color="#475569"))
ax3.annotate("", xy=(0.81, 0.58), xytext=(0.78, 0.38), arrowprops=dict(arrowstyle="->", lw=1.4, color="#475569"))
ax3.text(0.10, 0.28, "selected from CV candidates", ha="center", fontsize=9)
ax3.text(0.50, 0.28, "deployment-primary reference", ha="center", fontsize=9)
ax3.text(0.80, 0.28, "diagnostic larger-sample refit", ha="center", fontsize=9)
ax3.text(0.02, 0.92, "Deployment-oriented model states", fontsize=10, fontweight="bold")

fig


In [ ]:
fig_3_5_path = FIGURES_DIR / "fig_3_5_walkforward_model_states.png"
fig.savefig(fig_3_5_path, bbox_inches="tight")
fig_3_5_path


## Figure 3.6 — Metric hierarchy for deployment-oriented interpretation

**Why this format**
- ASCII is enough because this is a hierarchy of interpretation, not a measured plot.

**Metadata**
- Figure type: `conceptual`
- Evidence source: `thesis-only`
- Rendering method: `ASCII`
- Primary inputs: `co_om_thesis_enhanced.md`, metric names from `final_runs/*/final_report.csv`
- Acceptance check: AUC metrics as diagnostics, `gross_pnl_sum` as signal extraction, `n_trades` as turnover evidence, `pnl_sum` as primary economic outcome


**Generation body — final ASCII**

```text
                     DEPLOYMENT-ORIENTED METRIC HIERARCHY

                 ┌──────────────────────────────────────┐
                 │ Ranking diagnostics                  │
                 │  - dir_auc                           │
                 │  - trade_auc                         │
                 └──────────────────────────────────────┘
                                   │
                                   ▼
                 ┌──────────────────────────────────────┐
                 │ Pre-cost signal extraction           │
                 │  - gross_pnl_sum                     │
                 └──────────────────────────────────────┘
                                   │
                    ┌──────────────┴──────────────┐
                    ▼                             ▼
     ┌──────────────────────────────┐   ┌──────────────────────────────┐
     │ Turnover evidence            │   │ Selectivity / activity check │
     │  - n_trades                  │   │  - is the result supported   │
     │  - trade rate (optional)     │   │    by meaningful trading?    │
     └──────────────────────────────┘   └──────────────────────────────┘
                    │                             │
                    └──────────────┬──────────────┘
                                   ▼
                 ┌──────────────────────────────────────┐
                 │ Primary deployment outcome           │
                 │  - pnl_sum (post-cost net PnL)       │
                 └──────────────────────────────────────┘
```


## Figure 4.1 — Architecture comparison of `base_gnn`, `multigraph`, and `memorygraph`

**Why this format**
- Hybrid is best because the figure is conceptual but must stay faithful to actual architectural differences visible in the code.

**Metadata**
- Figure type: `hybrid`
- Evidence source: `code-grounded`
- Rendering method: `Prompt-first hybrid`
- Primary inputs: `models/base_gnn_pipeline.py`, `models/multigraph_pipeline.py`, `models/memorygraph_pipeline.py`, `co_om_thesis_enhanced.md`
- Acceptance check: early relation fusion vs relation-specific pathways vs recurrent node-edge memory, shared output heads


**Generation body — final image prompt**

```text
Create a publication-quality academic architecture comparison diagram for a master's thesis.

Title: "Architecture comparison of base_gnn, multigraph, and memorygraph"

Show three aligned columns:
- base_gnn: early relation fusion
- multigraph: relation-specific graph pathways before learned fusion
- memorygraph: recurrent node and edge memory states with graph interaction inside the loop

Shared bottom section:
- common output heads for trade relevance, direction, return, exit-related outputs
```


## Figure 4.2 — Recurrent node and edge memory update in `memorygraph`

**Why this format**
- Hybrid is necessary because the underlying mechanism is code-grounded but visually too complex for a purely auto-laid-out plot.

**Metadata**
- Figure type: `hybrid`
- Evidence source: `code-grounded`
- Rendering method: `Prompt-first hybrid`
- Primary inputs: `models/memorygraph_pipeline.py`, `co_om_thesis_enhanced.md`
- Acceptance check: edge memory from current edge/node states, node update from relation-specific edge context, recurrent loop visible


**Generation body — final image prompt**

```text
Create a clean academic recurrent-mechanism diagram for a graph neural network thesis.

Title: "Recurrent node and edge memory update in memorygraph"

Show:
1. inputs to edge memory update
2. edge memory update block
3. relation-specific edge context aggregation
4. node memory update block
5. updated node and edge states
6. explicit recurrent loop across time steps
```


## Figure 5.1 — Benchmark overview by frequency, graph family, and operator

**Why this format**
- Python is mandatory because this figure is a direct visual summary of benchmark results and should be generated from the result tables.

**Metadata**
- Figure type: `executable`
- Evidence source: `artifact-grounded`
- Rendering method: `Python`
- Primary inputs: `final_runs/*/final_report.csv`, optionally `*_final_summary.csv`
- Acceptance check: all 18 primary `last_CV` model-frequency configurations, grouped by frequency and family/operator, metric=`pnl_sum`, caution note about no significance testing


**Generation body**
- Extract `last_cv` rows only.
- Normalize labels into 18 comparable benchmark entries.
- Use grouped bar chart or heatmap.
- Add note: no uncertainty intervals / no formal dominance testing.

**Planned future code cells**
1. load benchmark summaries
2. normalize benchmark labels
3. render results overview
4. export figure


In [ ]:
benchmark_df = load_primary_benchmark_table(model_state="last_cv")
benchmark_df["family"] = benchmark_df["model_label"].str.extract(r"^(base-gnn|multi-gnn|memory-gnn)")
benchmark_df["operator_short"] = benchmark_df["model_label"].str.extract(r"(conv|mpnn)$")[0].str.upper()
benchmark_df["x_label"] = benchmark_df["family"].map({"base-gnn": "Base", "multi-gnn": "Multi", "memory-gnn": "Memory"}) + "\n" + benchmark_df["operator_short"]
benchmark_order = [
    "base-gnn-conv",
    "base-gnn-mpnn",
    "multi-gnn-conv",
    "multi-gnn-mpnn",
    "memory-gnn-conv",
    "memory-gnn-mpnn",
]
benchmark_df["order"] = benchmark_df["model_label"].map({label: i for i, label in enumerate(benchmark_order)})
benchmark_df.sort_values(["frequency", "order"])[["frequency", "model_label", "pnl_sum", "gross_pnl_sum", "n_trades"]]


In [ ]:
benchmark_panels = {
    freq: benchmark_df[benchmark_df["frequency"] == freq].sort_values("order").reset_index(drop=True)
    for freq in ["5min", "1min", "1sec"]
}


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.8), sharey=True, constrained_layout=True)
for ax, freq in zip(axes, ["5min", "1min", "1sec"]):
    panel = benchmark_panels[freq]
    bars = ax.bar(
        panel["x_label"],
        panel["pnl_sum"],
        color=panel["family"].map(FAMILY_COLORS),
        edgecolor="black",
        linewidth=0.5,
    )
    ax.axhline(0, color="black", linewidth=0.9)
    ax.set_title(freq)
    ax.set_ylabel("Net PnL (`pnl_sum`)")
    for bar, value in zip(bars, panel["pnl_sum"]):
        y = value + (0.01 if value >= 0 else -0.015)
        va = "bottom" if value >= 0 else "top"
        ax.text(bar.get_x() + bar.get_width() / 2, y, f"{value:.3f}", ha="center", va=va, fontsize=8)
    best_label = panel.loc[panel["pnl_sum"].idxmax(), "model_label"]
    ax.text(0.02, 0.95, f"best: {best_label}", transform=ax.transAxes, ha="left", va="top", fontsize=8, color="#334155")

fig.suptitle("Figure 5.1 — Benchmark overview by frequency, graph family, and operator", fontsize=13)
axes[0].text(
    -0.45,
    -0.32,
    "Note: visual summary of primary last_CV benchmark results; no uncertainty intervals or formal dominance testing are shown.",
    transform=axes[0].transAxes,
    fontsize=8,
    color="#475569",
)
fig


In [ ]:
fig_5_1_path = FIGURES_DIR / "fig_5_1_benchmark_overview.png"
fig.savefig(fig_5_1_path, bbox_inches="tight")
fig_5_1_path


## Figure 5.2 — Gross versus net PnL at `1sec`

**Why this format**
- Python is best because the figure is a direct comparison of reported metrics and trade counts.

**Metadata**
- Figure type: `executable`
- Evidence source: `artifact-grounded`
- Rendering method: `Python`
- Primary inputs: `final_runs/1sec-*/final_report.csv`
- Acceptance check: six 1sec models, `gross_pnl_sum` and `pnl_sum`, `n_trades`, clear `memory-gnn-conv` cost-drag story


**Generation body**
- Build paired bars for gross vs net PnL.
- Annotate or encode `n_trades`.
- Add note explaining cost burden dominance under extreme turnover.

**Planned future code cells**
1. load 1sec summaries
2. prepare gross/net/trade-count table
3. render comparison chart
4. export figure


In [ ]:
one_sec_df = benchmark_df[benchmark_df["frequency"] == "1sec"].copy().sort_values("order").reset_index(drop=True)
one_sec_df[["model_label", "gross_pnl_sum", "pnl_sum", "n_trades", "dir_auc", "trade_auc"]]


In [ ]:
one_sec_df["x_label"] = one_sec_df["model_label"].map({
    "base-gnn-conv": "Base\nConv",
    "base-gnn-mpnn": "Base\nMPNN",
    "multi-gnn-conv": "Multi\nConv",
    "multi-gnn-mpnn": "Multi\nMPNN",
    "memory-gnn-conv": "Memory\nConv",
    "memory-gnn-mpnn": "Memory\nMPNN",
})


In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 5.2), constrained_layout=True)
x = np.arange(len(one_sec_df))
width = 0.36

ax.bar(x - width / 2, one_sec_df["gross_pnl_sum"], width=width, color="#cbd5e1", edgecolor="black", linewidth=0.5, label="gross_pnl_sum")
ax.bar(x + width / 2, one_sec_df["pnl_sum"], width=width, color=one_sec_df["family"].map(FAMILY_COLORS), edgecolor="black", linewidth=0.5, label="pnl_sum")
ax.axhline(0, color="black", linewidth=0.9)
ax.set_xticks(x)
ax.set_xticklabels(one_sec_df["x_label"])
ax.set_ylabel("PnL")
ax.set_title("Figure 5.2 — Gross versus net PnL at 1sec")

for x_i, gross_value, net_value in zip(x, one_sec_df["gross_pnl_sum"], one_sec_df["pnl_sum"]):
    ax.text(x_i - width / 2, gross_value + 0.02, f"{gross_value:.3f}", ha="center", va="bottom", fontsize=8, rotation=90)
    ax.text(x_i + width / 2, net_value - 0.05 if net_value < 0 else net_value + 0.02, f"{net_value:.3f}", ha="center", va="top" if net_value < 0 else "bottom", fontsize=8, rotation=90)

ax2 = ax.twinx()
ax2.plot(x, one_sec_df["n_trades"], color="#7c2d12", marker="o", linewidth=1.8, label="n_trades")
ax2.set_ylabel("Number of trades", color="#7c2d12")
ax2.tick_params(axis="y", labelcolor="#7c2d12")
for x_i, trades in zip(x, one_sec_df["n_trades"]):
    ax2.text(x_i, trades + max(one_sec_df["n_trades"]) * 0.03, f"{int(trades)}", ha="center", va="bottom", fontsize=8, color="#7c2d12")

handles_1, labels_1 = ax.get_legend_handles_labels()
handles_2, labels_2 = ax2.get_legend_handles_labels()
ax.legend(handles_1 + handles_2, labels_1 + labels_2, loc="upper left", frameon=False)
ax.text(0.98, 0.96, "memory-gnn-conv: strongest gross signal but\nturnover overwhelms post-cost viability", transform=ax.transAxes, ha="right", va="top", fontsize=9, color="#7c2d12")

fig


In [ ]:
fig_5_2_path = FIGURES_DIR / "fig_5_2_gross_vs_net_1sec.png"
fig.savefig(fig_5_2_path, bbox_inches="tight")
fig_5_2_path


## Figure 5.3 — `last_CV` versus `final_refit` as deployment-oriented model states

**Why this format**
- Python is appropriate because the comparison already exists in artifact tables and should be shown as a paired state comparison, not just as a conceptual diagram.

**Metadata**
- Figure type: `executable`
- Evidence source: `artifact-grounded`
- Rendering method: `Python`
- Primary inputs: `final_runs/*/final_report.csv`, `final_runs/**/*final_holdout_model_comparison_summary.csv`
- Acceptance check: `last_CV` vs `final_refit`, deployment-primary role of `last_CV`, informative-but-non-primary role of `final_refit`, representative model cases


**Generation body**
- Prefer slope chart or paired bars.
- Use selected representative models if full matrix is too dense.
- Preserve semantic emphasis: deployment reference vs larger-sample diagnostic comparison.

**Planned future code cells**
1. load state-comparison artifacts
2. choose representative cases
3. render paired state comparison
4. export figure


In [ ]:
state_case_files = {
    "5min | base-gnn-conv": REPO_ROOT / "final_runs/5min-base-gnn/adaptive_conv/adaptive_conv_final_holdout_model_comparison_summary.csv",
    "1min | base-gnn-conv": REPO_ROOT / "final_runs/1min-base-gnn-conv/adaptive_conv/adaptive_conv_final_holdout_model_comparison_summary.csv",
    "1sec | memory-gnn-conv": REPO_ROOT / "final_runs/1sec-memory-gnn-conv/conv/conv_final_holdout_model_comparison_summary.csv",
    "5min | multi-gnn-conv": REPO_ROOT / "final_runs/5min-multi-gnn/dynamic_rel_conv/dynamic_rel_conv_final_holdout_model_comparison_summary.csv",
}

state_rows = []
for case_label, csv_path in state_case_files.items():
    df = pd.read_csv(csv_path)
    df = df[df["model_role"].isin(["last_cv_fold_model", "final_refit_model"])].copy()
    df["case_label"] = case_label
    df["state"] = df["model_role"].map({"last_cv_fold_model": "last_CV", "final_refit_model": "final_refit"})
    state_rows.append(df[["case_label", "state", "pnl_sum", "dir_auc", "trade_auc", "n_trades", "gross_pnl_sum"]])

state_comparison_df = pd.concat(state_rows, ignore_index=True)
state_case_order = list(state_case_files.keys())
state_comparison_df["case_order"] = state_comparison_df["case_label"].map({label: i for i, label in enumerate(state_case_order)})
state_comparison_df["state_order"] = state_comparison_df["state"].map({"last_CV": 0, "final_refit": 1})
state_comparison_df.sort_values(["case_order", "state_order"])


In [ ]:
state_plot_df = state_comparison_df.sort_values(["case_order", "state_order"]).reset_index(drop=True)
state_colors = {"last_CV": "#4c78a8", "final_refit": "#f58518"}
state_y = np.arange(len(state_case_order))[::-1]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5.6), constrained_layout=True)
for ax, metric, panel_title in zip(
    axes,
    ["pnl_sum", "dir_auc"],
    ["Economic outcome by model state", "Predictive ranking diagnostic by model state"],
):
    for i, case in enumerate(state_case_order):
        subset = state_plot_df[state_plot_df["case_label"] == case].sort_values("state_order")
        values = subset[metric].to_list()
        ax.plot(values, [state_y[i], state_y[i]], color="#94a3b8", linewidth=2.0, zorder=1)
        for _, row in subset.iterrows():
            ax.scatter(row[metric], state_y[i], s=90, color=state_colors[row["state"]], edgecolor="black", linewidth=0.4, zorder=3)
            text_dx = 0.008 if metric == "dir_auc" else 0.012
            ax.text(row[metric] + text_dx, state_y[i] + 0.07, f"{row[metric]:.3f}", fontsize=8, va="bottom")
    ax.set_yticks(state_y)
    ax.set_yticklabels(state_case_order)
    ax.set_title(panel_title)
    if metric == "pnl_sum":
        ax.axvline(0, color="black", linewidth=0.9)

axes[0].text(0.01, 1.05, "Figure 5.3 — `last_CV` versus `final_refit` as deployment-oriented model states", transform=axes[0].transAxes, fontsize=13, fontweight="bold", ha="left")
legend_handles = [
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=color, markeredgecolor="black", markersize=8, label=label)
    for label, color in state_colors.items()
]
axes[1].legend(handles=legend_handles, loc="lower right", frameon=False)
axes[0].text(
    0.01,
    -0.18,
    "Selected cases mirror the thesis discussion: two best shared-task models, one high-frequency stress example, and one informative multigraph refit case.",
    transform=axes[0].transAxes,
    fontsize=8,
    color="#475569",
)
fig


In [ ]:
fig_5_3_path = FIGURES_DIR / "fig_5_3_last_cv_vs_final_refit.png"
fig.savefig(fig_5_3_path, bbox_inches="tight")
fig_5_3_path


## Figure 6.1 — Deployment interpretation from prediction to post-cost evidence

**Why this format**
- ASCII works well because this is a reasoning chain rather than a data graphic.

**Metadata**
- Figure type: `conceptual`
- Evidence source: `thesis-only`
- Rendering method: `ASCII`
- Primary inputs: `co_om_thesis_enhanced.md`
- Acceptance check: predictive ranking, gross signal extraction, trade selectivity, cost adjustment, model-state stability, deployment-informative conclusion


**Generation body — final ASCII**

```text
 prediction quality
 (ranking diagnostics)
         │
         ▼
 gross signal extraction
 (`gross_pnl_sum` before cost)
         │
         ▼
 trade selectivity / turnover
 (`n_trades`, activity discipline)
         │
         ▼
 transaction-cost adjustment
 (gross edge must survive frictions)
         │
         ▼
 model-state stability
 (`last_CV` vs `final_refit` interpretation)
         │
         ▼
 deployment-informative evidence
 (`pnl_sum` after cost, with stable interpretation)
```


## Figure 7.1 — Future research roadmap for deployment-oriented graph LOB prediction

**Why this format**
- Manual-vector-first is the safest option because the roadmap is dense, conceptual, and taxonomy-sensitive; prompt can help as a draft, but should not be the authoritative final form.

**Metadata**
- Figure type: `conceptual`
- Evidence source: `thesis-only`
- Rendering method: `Manual-vector-first with prompt fallback`
- Primary inputs: `co_om_thesis_enhanced.md`
- Acceptance check: all seven roadmap directions present, thesis-faithful taxonomy, clear grouping, no generic fintech clichés


**Generation body — final image prompt fallback**

```text
Create an academic roadmap infographic for a master's thesis in graph-based market microstructure modelling.

Title: "Future research roadmap for deployment-oriented graph LOB prediction"

Central concept:
A roadmap centered on deployment-oriented graph LOB prediction.

Surrounding thematic branches:
1. turnover-aware learning
2. execution-aware evaluation
3. larger graph universes
4. selective memory mechanisms
5. regime robustness
6. uncertainty quantification
7. cost-sensitivity analysis

Important semantic constraint:
This figure is a future research agenda for a controlled benchmark thesis.
```


## Summary table

| Figure | Recommended rendering | Evidence source | Confidence |
|---|---|---|---|
| 1.1 | Hybrid (prompt-first) | Mixed | Medium |
| 1.2 | Python | Thesis-only | High |
| 3.1 | Python | Code-grounded | Medium |
| 3.2 | Python | Artifact-grounded | High |
| 3.3 | Hybrid (Python-primary) | Mixed | Medium |
| 3.4 | Hybrid (prompt-first) | Mixed | Medium |
| 3.5 | Python | Artifact-grounded | High |
| 3.6 | ASCII | Thesis-only | High |
| 4.1 | Hybrid (prompt-first) | Code-grounded | Medium |
| 4.2 | Hybrid (prompt-first) | Code-grounded | Medium |
| 5.1 | Python | Artifact-grounded | High |
| 5.2 | Python | Artifact-grounded | High |
| 5.3 | Python | Artifact-grounded | High |
| 6.1 | ASCII | Thesis-only | High |
| 7.1 | Manual-vector-first, prompt fallback | Thesis-only | Medium |


## Recommended implementation order

1. highest-confidence executable figures: 3.2, 3.5, 5.1, 5.2, 5.3
2. simple conceptual/executable figures: 1.2, 3.1
3. hybrid figures: 1.1, 3.3, 3.4, 4.1, 4.2
4. thesis-only conceptual visuals: 3.6, 6.1, 7.1
